# Evidence Selection

Picking the right chunks before the LLM sees them, with LangChain retrievers and a structured-output judge.

The primary objective of this demonstration is to isolate one variable at a time - WHICH chunks reach the prompt, not WHO writes the query and not HOW the embedder works. We will start by reproducing the most common production anti-pattern (**evidence dumping**) so the selection layer in the later sections lands as a measurable improvement, not an architectural opinion.

## Setup

Let's wire the model-agnostic stack and build a deliberately redundant corpus.

Now, we will
- assemble a 14-document corpus with intentional redundancy (three paraphrases of the same fact) and several distractor chunks,
- index it in an `InMemoryVectorStore` and define a single `QUESTION` whose answer requires three facts from three different docs,
- define a tiny `approx_tokens` helper so we can compare token cost across selection strategies later.

In [2]:
# !pip install -q langchain langchain-google-genai langchain-openai langchain-anthropic langgraph langchain-community

from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
from pydantic import BaseModel, Field
from typing import List, Literal
import os, json, time
# Keep the API keys for the models of choice in the loaded env file
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# os.environ['GEMINI_API_KEY']  # the variable for API key

llm   = init_chat_model('gemini-3.5-flash-lite', model_provider='google_genai', temperature=0)
embed = init_embeddings('sentence-transformers/all-MiniLM-L6-v2', provider='huggingface')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Now the corpus. The 14 chunks are adversarial-by-design: *d1 / d2 / d3* paraphrase the same Atlas-lead fact (a redundancy trap), *d4 / d5* hold the reports-to fact, *d6 / d7* hold the warehouse name, and *d10 - d14* are distractors that share vocabulary with the question without containing the answer. A naive top-k will burn its slots on the redundancy cluster and leave one of the required facts uncovered.

In [4]:
# 14-doc corpus engineered to expose evidence-dumping pathologies.
# d1, d2, d3   -> SAME Atlas-lead fact, three paraphrases (redundancy trap)
# d4, d5       -> reports-to fact (Priya -> Dinesh Kapoor)
# d6, d7       -> warehouse name (ATLAS_PROD on Snowflake)
# d8, d9       -> weakly related Atlas context (orchestration, ingestion)
# d10..d14     -> distractors that share vocabulary but not the answer
CORPUS = [
    {'id': 'd1',  'text': 'Project Atlas is led by Priya Raman, Principal Engineer, who joined the data platform team in 2022.'},
    {'id': 'd2',  'text': 'Priya Raman heads Project Atlas; she was promoted to Principal Engineer before taking the role.'},
    {'id': 'd3',  'text': 'The Atlas initiative technical lead is Priya Raman, a Principal Engineer on the data platform org.'},
    {'id': 'd4',  'text': 'Priya reports to Dinesh Kapoor, VP of Data Engineering, who owns the overall platform roadmap.'},
    {'id': 'd5',  'text': 'Dinesh Kapoor is VP of Data Engineering and is the skip-level for every Atlas sub-team lead.'},
    {'id': 'd6',  'text': 'The Atlas pipeline writes curated fact tables into the Snowflake warehouse ATLAS_PROD.'},
    {'id': 'd7',  'text': 'ATLAS_PROD on Snowflake is the destination warehouse for all Atlas batch and streaming jobs.'},
    {'id': 'd8',  'text': 'Atlas ingests from Kafka topics orders.v2 and inventory.v2 before landing in the warehouse.'},
    {'id': 'd9',  'text': 'The Atlas team uses dbt for transformations and Airflow for orchestration across environments.'},
    {'id': 'd10', 'text': 'Project Hermes is led by Arjun Mehta and focuses on real-time pricing, unrelated to Atlas.'},
    {'id': 'd11', 'text': 'Hermes writes to a separate Redshift cluster and reports to a different VP than Atlas.'},
    {'id': 'd12', 'text': 'The company uses Snowflake, BigQuery, and Redshift across different business units.'},
    {'id': 'd13', 'text': 'Data governance policies require PII redaction before any write to a production warehouse.'},
    {'id': 'd14', 'text': 'Priya Raman previously led Project Mercury, a deprecated ETL system retired in 2021.'},
]

docs = [Document(page_content=c['text'], metadata={'id': c['id']}) for c in CORPUS]
vectorstore = InMemoryVectorStore.from_documents(docs, embedding=embed)
print('Corpus size:', len(CORPUS))

Corpus size: 14


Now the question and the token-cost helper. `QUESTION` deliberately needs three facts from three different chunks (lead, reports-to, warehouse) so any selector that over-indexes on one cluster will visibly miss a fact in its answer.

In [5]:
def approx_tokens(text: str) -> int:
    """Rough token estimator: ~4 chars per token. Good enough for teaching
    token-economics across selection strategies; do NOT use this for billing."""
    return max(1, len(text) // 4)

# One question, three required facts from three disjoint chunks.
QUESTION = 'Who leads Project Atlas, who do they report to, and what warehouse does the Atlas pipeline write to?'
print('QUESTION:', QUESTION)

QUESTION: Who leads Project Atlas, who do they report to, and what warehouse does the Atlas pipeline write to?


## Evidence dumping baseline

Let's start with the production anti-pattern: retrieve a wide top-k and stuff every chunk into the prompt. This is the path of least resistance - fewer lines, no selection logic, and *more context can't hurt* feels safe.

Now, we will
- pull the top-10 nearest chunks from the vector store (a wide net by design),
- concatenate them into one prompt block, labelled `[d?]` so the model could cite,
- invoke the LLM once and record the chunks, the approximate prompt tokens, and the latency,
- read the failure mode: bloated token cost, **lost in the middle** behaviour where the answer chunk gets buried, and distractor leakage where the LLM picks up an unrelated fact (e.g. confuses *Hermes* with *Atlas*) because every chunk in the prompt looks equally authoritative.

In [6]:
def format_context(docs_in: List[Document]) -> str:
    """Serialise selected docs into a numbered context block. Kept identical across
    every strategy in this notebook so quality deltas are attributable to SELECTION,
    not to prompt wording."""
    return '\n'.join(f"[{d.metadata['id']}] {d.page_content}" for d in docs_in)

Now the prompt template and the dumping baseline itself. The prompt lives at module level as an UPPER_CASE constant so every selection strategy in this notebook reuses **the exact same** instruction text.

In [15]:
def as_text(content) -> str:
    """Some providers/models return .content as a string, others as a list of
    content blocks (e.g. [{'type': 'text', 'text': ...}]). Normalize to plain text."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return ''.join(
            block.get('text', '') for block in content
            if isinstance(block, dict) and block.get('type') == 'text'
        )
    return str(content)

In [16]:
ANSWER_PROMPT = """Answer the question using ONLY the evidence below.
If the evidence does not contain the answer, say so briefly.

EVIDENCE:
{ctx}

QUESTION: {q}
ANSWER:"""

# Pattern: retrieve large k -> dump everything into the prompt -> LLM
# There is NO selection layer between retrieval and generation.
def dump_everything(question: str, k: int = 10) -> dict:
    """Evidence-dumping baseline. Pulls a wide top-k and concatenates every chunk
    into the prompt. The LLM has to filter signal from noise on its own — which it
    does badly when the prompt is long (lost in the middle) and full of paraphrases."""
    hits = vectorstore.similarity_search(question, k=k)
    ctx  = format_context(hits)
    prompt = ANSWER_PROMPT.format(ctx=ctx, q=question)
    t0 = time.time()
    print("%%%%%%%%%%%%%%%%%%%%%%%%")
    print(ctx)
    print("-------------")
    answer = as_text(llm.invoke(prompt).content)
    print(answer)
    print("%%%%%%%%%%%%%%%%%%%%%%%%")
    return {
        'strategy': 'dump',
        'ids':      [d.metadata['id'] for d in hits],
        'tokens':   approx_tokens(prompt),
        'latency':  round(time.time() - t0, 2),
        'answer':   answer,
    }

Time to invoke. We are expecting a long prompt, an inflated token count, and an answer that either hedges, buries the warehouse fact, or absorbs a distractor (the *lost-in-the-middle* effect on multi-fact questions).

In [13]:
QUESTION

'Who leads Project Atlas, who do they report to, and what warehouse does the Atlas pipeline write to?'

In [17]:
dump_result = dump_everything(QUESTION, k=10)

print('=== DUMP BASELINE ===')
print('Retrieved ids :', dump_result['ids'])
print('Prompt tokens~:', dump_result['tokens'])
print('Latency       :', dump_result['latency'], 's')
print('Answer        :')
print(dump_result['answer'][:600] + ('...' if len(dump_result['answer']) > 600 else ''))

%%%%%%%%%%%%%%%%%%%%%%%%
[d1] Project Atlas is led by Priya Raman, Principal Engineer, who joined the data platform team in 2022.
[d7] ATLAS_PROD on Snowflake is the destination warehouse for all Atlas batch and streaming jobs.
[d3] The Atlas initiative technical lead is Priya Raman, a Principal Engineer on the data platform org.
[d6] The Atlas pipeline writes curated fact tables into the Snowflake warehouse ATLAS_PROD.
[d9] The Atlas team uses dbt for transformations and Airflow for orchestration across environments.
[d8] Atlas ingests from Kafka topics orders.v2 and inventory.v2 before landing in the warehouse.
[d10] Project Hermes is led by Arjun Mehta and focuses on real-time pricing, unrelated to Atlas.
[d11] Hermes writes to a separate Redshift cluster and reports to a different VP than Atlas.
[d2] Priya Raman heads Project Atlas; she was promoted to Principal Engineer before taking the role.
[d12] The company uses Snowflake, BigQuery, and Redshift across different business units

/home/navneet/miniconda3/envs/gen311/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided evidence:
* **Who leads Project Atlas:** Priya Raman (Principal Engineer).
* **Who do they report to:** The provided evidence does not contain information about who Priya Raman reports to.
* **What warehouse does the Atlas pipeline write to:** ATLAS_PROD on Snowflake.
%%%%%%%%%%%%%%%%%%%%%%%%
=== DUMP BASELINE ===
Retrieved ids : ['d1', 'd7', 'd3', 'd6', 'd9', 'd8', 'd10', 'd11', 'd2', 'd12']
Prompt tokens~: 304
Latency       : 0.94 s
Answer        :
Based on the provided evidence:
* **Who leads Project Atlas:** Priya Raman (Principal Engineer).
* **Who do they report to:** The provided evidence does not contain information about who Priya Raman reports to.
* **What warehouse does the Atlas pipeline write to:** ATLAS_PROD on Snowflake.


So **evidence dumping** has a real cost. The prompt is several times bigger than it needs to be, three of the ten chunks (*d1 / d2 / d3*) are paraphrases of the same fact, and the model now has to do the selection job inside its own attention pass, which is exactly the failure mode the rest of this notebook fixes.

## MMR for diversity

Now let's add a real selection layer. **Maximal Marginal Relevance** picks chunks that are relevant to the query AND unlike the chunks already chosen. We are using this to break the *d1 / d2 / d3* redundancy cluster.

Now, we will
- build an MMR retriever via `vectorstore.as_retriever(search_type='mmr', ...)`,
- set `k=4` (final picks), `fetch_k=12` (wider candidate pool), `lambda_mult=0.5` (balanced relevance / diversity),
- compare top-k similarity selection against MMR selection on the SAME question,
- confirm MMR returns one paraphrase from the redundancy cluster, plus chunks covering the reports-to and warehouse facts.

In [18]:
# Top-k similarity retriever (k=4) — for direct comparison with MMR.
topk_retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4},
)

# MMR retriever: same final k, but applies MMR over a wider fetch_k pool.
# lambda_mult: 1.0 = pure relevance (= top-k), 0.0 = pure diversity. 0.5 is balanced.
mmr_retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 4, 'fetch_k': 12, 'lambda_mult': 0.5},
)

MMR's three knobs are the only tuning surface most teams need:

- **`k`**: number of chunks the retriever finally returns.
- **`fetch_k`**: wider candidate pool MMR runs over (typically 2-3x `k`). Smaller and MMR has nothing to choose from; larger wastes embedding bandwidth.
- **`lambda_mult`**: relevance vs diversity. 1.0 collapses to top-k, 0.0 ignores the question entirely. 0.5 to 0.6 is the production default.

In [19]:
def ids_of(docs_in: List[Document]) -> List[str]:
    """Pull out doc ids for side-by-side inspection of selection strategies."""
    return [d.metadata['id'] for d in docs_in]

topk_docs = topk_retriever.invoke(QUESTION)
mmr_docs  = mmr_retriever.invoke(QUESTION)

print('---- TOP-K (similarity, k=4) ----')
print('ids:', ids_of(topk_docs))
for d in topk_docs:
    print(f"  [{d.metadata['id']}] {d.page_content[:90]}...")

print('\n---- MMR (k=4, fetch_k=12, lambda_mult=0.5) ----')
print('ids:', ids_of(mmr_docs))
for d in mmr_docs:
    print(f"  [{d.metadata['id']}] {d.page_content[:90]}...")

---- TOP-K (similarity, k=4) ----
ids: ['d1', 'd7', 'd3', 'd6']
  [d1] Project Atlas is led by Priya Raman, Principal Engineer, who joined the data platform team...
  [d7] ATLAS_PROD on Snowflake is the destination warehouse for all Atlas batch and streaming job...
  [d3] The Atlas initiative technical lead is Priya Raman, a Principal Engineer on the data platf...
  [d6] The Atlas pipeline writes curated fact tables into the Snowflake warehouse ATLAS_PROD....

---- MMR (k=4, fetch_k=12, lambda_mult=0.5) ----
ids: ['d1', 'd8', 'd6', 'd13']
  [d1] Project Atlas is led by Priya Raman, Principal Engineer, who joined the data platform team...
  [d8] Atlas ingests from Kafka topics orders.v2 and inventory.v2 before landing in the warehouse...
  [d6] The Atlas pipeline writes curated fact tables into the Snowflake warehouse ATLAS_PROD....
  [d13] Data governance policies require PII redaction before any write to a production warehouse....


Top-k will typically pick multiple of *d1 / d2 / d3* (paraphrase cluster) and miss either the reports-to or the warehouse fact. MMR breaks that cluster: one Atlas-lead chunk, then the geometry of the remaining candidates pushes the next picks toward the disjoint fact clusters.

## LLM-as-judge filtering

Now let's add a semantic filter. MMR handles diversity on embedding geometry; it cannot detect negation, conditionals, or entity-type mismatches. An **LLM-as-judge** scores each candidate against the question with full language reasoning, and we keep only the ones it explicitly approves.

Now, we will
- define a tiny Pydantic schema `EvidenceVerdict` with `keep: bool` and a `reason: str`,
- bind it to the LLM via `with_structured_output(EvidenceVerdict)` so the verdict is machine-readable,
- pull a wider MMR pool of 8 candidates and judge each one,
- print every per-chunk verdict, then keep only the ones where `keep=True`.

In [20]:
class EvidenceVerdict(BaseModel):
    """Per-candidate verdict from the LLM judge. Using `with_structured_output`
    makes the decision MACHINE-READABLE so we can filter in pure Python without
    regex-parsing free text."""
    keep:   bool = Field(description='True if this snippet directly helps answer the question.')
    reason: str  = Field(description='One short sentence explaining the verdict.')

The schema is the filter. Two fields, both required, both typed

In [21]:
JUDGE_PROMPT = """You are an evidence judge for a retrieval-augmented system.

QUESTION: {q}

SNIPPET: {s}

Decide whether this snippet directly helps answer the question.
Return keep=true only if the snippet contributes a fact the answer needs.
Reject snippets that are off-topic, weakly related, or only restate background.
"""

judge_llm = llm.with_structured_output(EvidenceVerdict)

def llm_judge_filter(question: str, candidates: List[Document]) -> List[Document]:
    """Loop over candidates, ask the LLM to vote keep / drop on each, return keepers only.
    One LLM call per candidate — bound the candidate set with MMR or top-N FIRST."""
    keepers: List[Document] = []
    for d in candidates:
        verdict: EvidenceVerdict = judge_llm.invoke(JUDGE_PROMPT.format(q=question, s=d.page_content))
        mark = 'KEEP' if verdict.keep else 'DROP'
        print(f"  ---- [{d.metadata['id']}] {mark} ---- {verdict.reason[:120]}")
        if verdict.keep:
            keepers.append(d)
    return keepers

Now we exercise the judge over a wider MMR pool. Per-chunk verdicts double as a free debugging log — the *reason* field tells us exactly why the LLM dropped a distractor.

In [22]:
judge_pool_retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 8, 'fetch_k': 12, 'lambda_mult': 0.5},
)
judge_pool = judge_pool_retriever.invoke(QUESTION)
print("^^^^^^^^^^^^^^^^^")
print(judge_pool)
print("^^^^^^^^^^^^^^^^^")
print('=== JUDGE VERDICTS ===')
judged_docs = llm_judge_filter(QUESTION, judge_pool)

print('\n=== KEPT IDS ===')
print(ids_of(judged_docs))

^^^^^^^^^^^^^^^^^
[Document(id='64659ba3-a9fe-46ac-bc53-54184c825778', metadata={'id': 'd1'}, page_content='Project Atlas is led by Priya Raman, Principal Engineer, who joined the data platform team in 2022.'), Document(id='e1813e91-918e-4c73-9904-2547da003a4f', metadata={'id': 'd8'}, page_content='Atlas ingests from Kafka topics orders.v2 and inventory.v2 before landing in the warehouse.'), Document(id='5622e0ee-86bb-4204-ac7e-9452cc6455e7', metadata={'id': 'd6'}, page_content='The Atlas pipeline writes curated fact tables into the Snowflake warehouse ATLAS_PROD.'), Document(id='802ee4cf-5d81-4532-80d5-30ded86dfcd3', metadata={'id': 'd13'}, page_content='Data governance policies require PII redaction before any write to a production warehouse.'), Document(id='ad5ece3c-72ad-44da-9dd1-6e834fda8c1d', metadata={'id': 'd11'}, page_content='Hermes writes to a separate Redshift cluster and reports to a different VP than Atlas.'), Document(id='f50fbeba-a1d9-42bf-b546-d3a15d0a6d13', metadata={

/home/navneet/miniconda3/envs/gen311/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  ---- [d1] KEEP ---- The snippet identifies Priya Raman as the leader of Project Atlas, which directly answers part of the question.


/home/navneet/miniconda3/envs/gen311/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  ---- [d8] DROP ---- The snippet mentions the warehouse used by the Atlas pipeline, but does not provide information on who leads Project Atl


...

  ---- [d9] DROP ---- The snippet mentions the Atlas team and tools, but does not provide the project lead, reporting structure, or destinatio


/home/navneet/miniconda3/envs/gen311/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  ---- [d10] DROP ---- The snippet discusses Project Hermes, which is unrelated to Project Atlas.


/home/navneet/miniconda3/envs/gen311/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  ---- [d12] DROP ---- The snippet only mentions data warehouses used by the company but does not provide information about Project Atlas, its 

=== KEPT IDS ===
['d1', 'd6']


## Contextual compression for extractive selection

Sometimes the right move is not to drop chunks but to **shorten** them. Pull only the question-relevant sentences from each. 

Now, we will
- build an extractor from the same `llm` we have been using,
- wrap it around the MMR retriever,
- invoke it on the question and print the compressed snippets,
- confirm each surviving snippet is shorter than the original chunk yet still carries the load-bearing fact.

In [23]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

def extract_relevant(doc, query):
    prompt = f"""Extract ONLY verbatim spans relevant to the query.

Rules:
- Do NOT summarise
- Do NOT explain
- If nothing is relevant, return NOTHING (empty string)

Query:
{query}

Document:
{doc.page_content}
"""
    response = llm.invoke(prompt)
    text = response.content.strip()

    return Document(
        page_content=text,
        metadata=doc.metadata
    )

def compress_documents(docs, query):
    return [
        extract_relevant(d, query)
        for d in docs
        if d.page_content.strip()
    ]

In [50]:
def get_compressed_docs(query: str):
    docs = mmr_retriever.invoke(query)
    return compress_documents(docs, query)

compressed_docs = get_compressed_docs(QUESTION)

print('=== COMPRESSED SNIPPETS ===')
for d in compressed_docs:
    src_id = d.metadata.get('id', '?')
    print(f"---- [{src_id}] ----")
    print(d.page_content[:240] + ('...' if len(d.page_content) > 240 else ''))

=== COMPRESSED SNIPPETS ===
---- [d1] ----
Project Atlas is led by Priya Raman, Principal Engineer.
---- [d8] ----

---- [d6] ----
The Atlas pipeline writes curated fact tables into the Snowflake warehouse ATLAS_PROD.
---- [d13] ----



## Side by side: `dump_everything` vs `select_then_answer`

Time to put the layers together and quantify the delta. Same question, same retriever family, same prompt template. The only thing that varies is whether selection happens before the LLM sees the evidence.

Now, we will
- assemble `select_then_answer(question)` as the cascade `MMR -> EvidenceVerdict judge -> generate`,
- run it on the same `QUESTION` we used for the dumping baseline,
- print a small table contrasting strategy, chunks, prompt tokens, and latency,
- read both answers back to back and confirm the selected version is more complete at a fraction of the token cost.

In [26]:
def select_then_answer(question: str) -> dict:
    """Cascade: MMR widens the candidate pool with diversity, the EvidenceVerdict judge
    filters out distractors, and only the survivors land in the answer prompt.
    The prompt template is identical to dump_everything — that isolation is what lets us
    attribute the delta purely to SELECTION."""
    pool = judge_pool_retriever.invoke(question)
    keepers = llm_judge_filter(question, pool)
    ctx = format_context(keepers)
    prompt = ANSWER_PROMPT.format(ctx=ctx, q=question)
    t0 = time.time()
    answer = as_text(llm.invoke(prompt).content)
    return {
        'strategy': 'select',
        'ids':      ids_of(keepers),
        'tokens':   approx_tokens(prompt),
        'latency':  round(time.time() - t0, 2),
        'answer':   answer,
    }

print('=== SELECT-THEN-ANSWER (judge log) ===')
select_result = select_then_answer(QUESTION)

=== SELECT-THEN-ANSWER (judge log) ===


/home/navneet/miniconda3/envs/gen311/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  ---- [d1] KEEP ---- The snippet identifies that Priya Raman leads Project Atlas, which directly answers part of the question.


/home/navneet/miniconda3/envs/gen311/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  ---- [d8] DROP ---- The snippet mentions the warehouse used by the Atlas pipeline, but lacks information about who leads Project Atlas or wh


...

/home/navneet/miniconda3/envs/gen311/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  ---- [d10] DROP ---- The snippet discusses Project Hermes, which is unrelated to Project Atlas.


/home/navneet/miniconda3/envs/gen311/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


  ---- [d12] DROP ---- The snippet only mentions data warehouses used by the company but does not state who leads Project Atlas, who they repor


/home/navneet/miniconda3/envs/gen311/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Now we line up the two strategies. The `dump` row should have ~10 chunks and a token count several times larger than `select`; the `select` row should cover lead + reports-to + warehouse with a tighter answer.

In [27]:
rows = [
    ('dump',   len(dump_result['ids']),   dump_result['tokens'],   dump_result['latency']),
    ('select', len(select_result['ids']), select_result['tokens'], select_result['latency']),
]

print('=== STRATEGY COMPARISON ===')
print(f"{'strategy':<10}{'chunks':>8}{'tokens~':>10}{'latency_s':>12}")
for name, c, t, lat in rows:
    print(f'{name:<10}{c:>8}{t:>10}{lat:>12}')

print('\n=== DUMP ANSWER ===')
print(dump_result['answer'][:400] + ('...' if len(dump_result['answer']) > 400 else ''))

print('\n=== SELECT ANSWER ===')
print(select_result['answer'][:400] + ('...' if len(select_result['answer']) > 400 else ''))

=== STRATEGY COMPARISON ===
strategy    chunks   tokens~   latency_s
dump            10       304        0.94
select           2       109        0.68

=== DUMP ANSWER ===
Based on the provided evidence:
* **Who leads Project Atlas:** Priya Raman (Principal Engineer).
* **Who do they report to:** The provided evidence does not contain information about who Priya Raman reports to.
* **What warehouse does the Atlas pipeline write to:** ATLAS_PROD on Snowflake.

=== SELECT ANSWER ===
Based on the provided evidence:
* **Project Atlas is led by:** Priya Raman, Principal Engineer.
* **Who they report to:** The evidence does not contain this answer.
* **Warehouse the Atlas pipeline writes to:** Snowflake warehouse ATLAS_PROD.


**WHICH not WHAT.** Both strategies use the same retriever family, the same prompt template, and the same LLM. The accuracy and cost delta is purely **which chunks reach the prompt**.


### Tuning evidence selection in production
- Start MMR at `lambda_mult=0.5` and sweep plus-or-minus 0.2 on a labelled eval set; `fetch_k` should sit at 2-3x `k`.
- Cascade selectors: cheap retriever (top-N or MMR) narrows the pool, then the LLM judge or compressor refines. Never run the LLM judge over the full corpus.
- Use a cheap model (`gemini-2.5-flash`, `gpt-4o-mini`) for the judge / compressor and reserve the bigger generator for the final answer call.
- Keep `temperature=0` on every `with_structured_output` call. Higher temperature inflates the rate of malformed `EvidenceVerdict` responses.

### Common pitfalls in evidence dumping and selection
- Long prompts trigger the **lost in the middle** failure mode: the answer chunk gets buried and the LLM weighs distractors equally. Selection is the fix; longer context windows are not.
- Distractors that share vocabulary with the question (here, *Hermes* shares *project / leads / VP*) leak into dumped prompts and corrupt the answer. The judge catches them; pure cosine similarity does not.
- Embedder-only diversity (MMR) misses negation, conditionals, and entity-type mismatches. Layer an LLM judge on top when those failure modes show up in your eval set.
- Token budget is a quality lever, not just a cost lever. Halving the prompt often improves answer accuracy because the surviving chunks are denser per token.